In [1]:
## PyG Graph with Mesh Nodes
from pathlib import Path
from typing import Dict, List, Optional, Tuple
import glob
import numpy as np
from scipy.spatial import KDTree
from scipy.stats import norm
import torch
from torch_geometric.data import Data
from torch_geometric.utils import is_undirected, to_undirected
import pandas as pd

torch.set_default_dtype(torch.float32)
device = torch.device('cpu')

def stiffness_to_node_adj_edge_index(K: torch.Tensor, num_nodes: int, dof_per_node: int = 3) -> torch.Tensor: #mesh-mesh nodes
    K = K.coalesce()
    dof_rows, dof_cols = K.indices()
    node_rows = torch.div(dof_rows, dof_per_node, rounding_mode="floor")
    node_cols = torch.div(dof_cols, dof_per_node, rounding_mode="floor")
    ei = torch.stack([node_rows.long(), node_cols.long()], dim=0)
    mask = ei[0] != ei[1]
    ei = ei[:, mask]
    ei = torch.unique(ei, dim=1)
    return ei

def incidence_edges_from_conn(conn: np.ndarray, nodes: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    Nc, Nv = conn.shape
    c_idx = np.repeat(np.arange(Nc, dtype=np.int64), Nv)
    n_idx = conn.reshape(-1)
    edge_index = torch.from_numpy(np.vstack([c_idx, n_idx])).long()
    node_tensor = nodes.to(edge_index.device)
    conn_tensor = torch.from_numpy(conn).long().to(edge_index.device)
    #edge attribute -> mesh to element centroid in xyz
    elem_nodes = node_tensor[conn_tensor]
    centroids = elem_nodes.mean(dim=1, keepdim=True)
    rel_disp = (elem_nodes - centroids).reshape(-1, elem_nodes.size(-1))

    return edge_index, rel_disp

def mesh_edges_from_conn(conn: torch.Tensor) -> torch.Tensor:
    conn = conn.long()                             # [E, nverts]
    hex_edges = torch.tensor([
        [0,1], [1,2], [2,3], [3,0],     # bottom face
        [4,5], [5,6], [6,7], [7,4],     # top face
        [0,4], [1,5], [2,6], [3,7],     # vertical edges
    ])
    pairs = conn[:,hex_edges]          # [E, 12, 2]
    pairs = pairs.reshape(-1, 2)
    pairs = torch.unique(torch.sort(pairs, dim=1).values, dim=0).T
    edge_index = pairs
    return edge_index

def mesh_edges_nearest(nodes: torch.Tensor) -> torch.Tensor:
    # build mesh connections from nearest neighbors
    num_nodes = int(nodes.size(0))
    if num_nodes < 2:
        return torch.empty((2, 0), dtype=torch.long, device=nodes.device)

    coords = nodes.detach().cpu().numpy()
    k = min(8, num_nodes - 1)
    _, nn_idx = KDTree(coords).query(coords, k=k + 1, workers=-1)
    neighbors = nn_idx[:, 1:].reshape(-1)  # drop self
    src = np.repeat(np.arange(num_nodes, dtype=np.int64), k)

    pairs = torch.from_numpy(np.stack([src, neighbors], axis=1))
    pairs = torch.sort(pairs, dim=1).values
    pairs = torch.unique(pairs, dim=0)
    return pairs.t().to(nodes.device)

def nodes_topology_info(nodes: torch.Tensor) -> torch.Tensor:
    # build 3D descriptors of nearest neighbors, fitting into a distribution
    coords = nodes.detach().cpu().numpy()  # [N, 3]
    num_nodes = coords.shape[0]

    k = min(125, num_nodes - 1)  # cap k to available neighbors
    _, nn_idx = KDTree(coords).query(coords, k=k + 1, workers=-1)  # include self
    neighbor_idx = nn_idx[:, 1:]  # drop self index, shape (N, k)

    neighbor_coords = coords[neighbor_idx]             # (N, k, 3)
    rel = neighbor_coords - coords[:, None, :]         # relative vectors
    mu = torch.from_numpy(rel.mean(axis=1))                              # (N, 3)
    sigma = torch.from_numpy(rel.std(axis=1, ddof=0))                    # (N, 3)
    return torch.cat([mu, sigma], dim=1)


# Connectivity
# mesh nodes <-> mesh nodes
# mesh nodes <-> element nodes

def data_to_graph(path:str,device):
    data = Data()
    simdata = torch.load(path,weights_only=False)
    conn = simdata["elements"]
    conn = conn.cpu().numpy() if isinstance(conn, torch.Tensor) else np.asarray(conn)
    nodes = simdata['nodes']
    top_desc = nodes_topology_info(nodes)
    #c2n_ei, c2n_w = incidence_edges_from_conn(conn,nodes)  # [2, E_cn], [E_cn, 1]
    
    # mesh node properties: positions, forces, BC, dirichlet displacement
    data.pos = nodes #                                             [N,3]
    data.bc = simdata['boundary']
    data.f_ext = simdata['ext_forces'] #forces in timeseries format     [T,N,3]
    data.tpl = top_desc #nearest-neighbor topology descriptors    [N,6]

    # target properties
    # mesh: displacement over time
    data.f_ts = simdata['forces'] #forces in timeseries format     [T,N,3]
    data.u_ts = simdata['u_history']

    # mesh-mesh: distance
    #ei = stiffness_to_node_adj_edge_index(simdata["stiffness"], num_nodes=nodes.size(0), dof_per_node=nodes.size(1))
    ei = mesh_edges_from_conn(simdata["elements"]) # [2,N]
    #ei = mesh_edges_nearest(nodes)
    src, dst = ei
    #edge_mat = one_hot(vocab, device=device).unsqueeze(0).repeat(int(ei.shape[1]), 1)
    disp = (nodes[dst] - nodes[src]).float()
    data.edge_index = ei
    data.edge_attr = disp
    data = data.to(device)
    return data

#data_to_graph('../base/torchfem_dataset/processed/simulation_dump_3.pt','../base/torchfem_dataset/processed/mat.csv',device)

def generate_dataset(data_dir:str):
    device = torch.device('cpu')
    file = data_dir
    samples = []
    #for file in files:
        #if filename > 50:
        #    continue
    data = data_to_graph(file,device)
    samples.append(data)
    print(file)

    print(len(samples))
    return samples

dataset = generate_dataset('../datasets/simple_beam/sim_2.pt')

ea = dataset[0].edge_attr
edge_index = dataset[0].edge_index
edge_index,ea = to_undirected(edge_index,edge_attr=ea)
print('Undirected?' if is_undirected(edge_index) else 'Directed!')
dataset[0].edge_attr = ea
dataset[0].edge_index = edge_index

#dataset = generate_dataset('../torchfem_dataset/simple_beam_reduced/sim_2.pt')
#torch.save(dataset, "../datasets/simple_beam/test_dataset.pt")

../datasets/simple_beam/sim_2.pt
1
Undirected?


In [2]:
# Load Dataset

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.utils import coalesce
from torch_geometric.loader import DataLoader
from torch_geometric.data import HeteroData
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree, Delaunay
import time

def build_laststep_features(data,dtype=torch.float32):
    ## Collapse to peak load timestep
    #print(data.keys)
    len = data.f_ext.shape[0]
    t = int(len-1)
    # Nodes: x = [pos, bc, f_ext[-1], f_int[-1]]  (no leakage of u_ts into x)
    #pos   = data['nodes'].pos.to(dtype) #no need to include
    bc    = data.bc
    f_ext = torch.Tensor(data.f_ext[t]).to(dtype)
    data.fext = torch.Tensor(data.f_ext[t]).to(dtype)
    data.x = torch.cat([bc, f_ext], dim=-1).to(dtype)


    # Target: nodes â†’ u_ts[-1] (3D)
    data.y_u = data.u_ts[t].to(dtype)
    data.y_fint = data.f_ts[t].to(dtype)

    # (Optional) free large tensors you won't use further to save RAM/VRAM
    del data.bc, data.u_ts, data.f_ext, data.f_int#, data['nodes'].pos
    del data.f_ts

    #print(data.keys)

    return data

class StandardScaler:
    def __init__(self):
        self.node_stats = {}
        self.edge_stats = {}

    def fit(self, dataset):
        node_x = torch.cat([d.x[:,3:].float() for d in dataset],dim=0)
        node_f = torch.cat([d.y_fint.float() for d in dataset],dim=0)
        node_u = torch.cat([d.y_u.float() for d in dataset],dim=0)
        node_c_f = torch.cat([d.coarse_features[:,3:6].float() for d in dataset],dim=0)
        edge_c_l = torch.cat([d.edge_attr_coarse.float() for d in dataset],dim=0)

        # compute mean/std
        self.node_stats['nodes_x'] = {
            "mean": torch.zeros_like(node_x.mean(dim=0, keepdim=True)),
            "std":  node_x.std(dim=0, keepdim=True) + 1e-8}
        self.node_stats['nodes_f'] = {
            "mean": torch.zeros_like(node_f.mean(dim=0, keepdim=True)),
            "std":  node_f.std(dim=0, keepdim=True) + 1e-8}
        self.node_stats['nodes_u'] = {
            "mean": torch.zeros_like(node_u.mean(dim=0, keepdim=True)),
            "std":  node_u.std(dim=0, keepdim=True) + 1e-8}
        self.node_stats['nodes_c'] = {
            "mean": torch.zeros_like(node_c_f.mean(dim=0, keepdim=True)),
            "std":  node_c_f.std(dim=0, keepdim=True) + 1e-8}
        self.node_stats['edges_c'] = {
            "mean": torch.zeros_like(edge_c_l.mean(dim=0, keepdim=True)),
            "std":  edge_c_l.std(dim=0, keepdim=True) + 1e-8}

    def transform(self, data: torch.Tensor):
        # apply normalization
        x = data.x[:,3:].float()
        m_x = self.node_stats['nodes_x']["mean"]
        s_x = self.node_stats['nodes_x']["std"]
        data.x[:,3:] = (x - m_x) / s_x

        y_f = data.y_fint.float()
        m_f = self.node_stats['nodes_f']["mean"]
        s_f = self.node_stats['nodes_f']["std"]
        data.y_fint = (y_f - m_f) / s_f
        
        y_u = data.y_u.float()
        m_u = self.node_stats['nodes_u']["mean"]
        s_u = self.node_stats['nodes_u']["std"]
        data.y_u = (y_u - m_u) / s_u
        
        c_f = data.coarse_features[:,3:6].float()
        m_c_f = self.node_stats['nodes_c']["mean"]
        s_c_f = self.node_stats['nodes_c']["std"]
        data.coarse_features[:,3:6] = (c_f - m_c_f) / s_c_f
        
        c_e = data.edge_attr_coarse.float()
        m_e = self.node_stats['edges_c']["mean"]
        s_e = self.node_stats['edges_c']["std"]
        data.edge_attr_coarse = (c_e - m_e) / s_e

        return data
    
    def inverse_transform(self, data: torch.Tensor):
        # Nodes
        m, s = self.node_stats["nodes_u"]["mean"], self.node_stats["nodes_u"]["std"]
        data = data * s + m

        return data


def split_dataset(dataset, val_ratio=0.1, shuffle=True):
    n = len(dataset)
    idx = torch.randperm(n) if shuffle else torch.arange(n)
    n_val = max(1, int(n * val_ratio))
    val_idx = idx[:n_val].tolist()
    train_idx = idx[n_val:].tolist()
    train_set = [dataset[i] for i in train_idx]
    val_set   = [dataset[i] for i in val_idx]
    return train_set, val_set

# preprocess first
#train_set, val_set = split_dataset(dataset_t, val_ratio=0.1)

def minmax_norm(t):
    #t = t.abs()
    t_min = t.amin(dim=(0, 1), keepdim=True)   # shape (1, 1, 3)
    t_max = t.amax(dim=(0, 1), keepdim=True)   # shape (1, 1, 3)
    return (t - t_min) / (t_max - t_min + 1e-8)

def coarsen(data):
    # sample points spatially
    pts = data.pos
    frac = 0.05
    N = pts.shape[0] # number of points
    M = max(1, int(np.round(frac * N))) # number of selected points
    rng = np.random.default_rng(42)
    selected_idx = np.empty(M, dtype=int)

    start = rng.integers(N)
    selected_idx[0] = start
    diff = pts - pts[start]
    min_dist2 = np.einsum("ij,ij->i", diff, diff)
    min_dist2[start] = 0.0

    for i in range(1, M): # random elimination
        idx = np.argmax(min_dist2)
        selected_idx[i] = idx
        diff = pts - pts[idx]
        dist2 = np.einsum("ij,ij->i", diff, diff)
        min_dist2 = np.minimum(min_dist2, dist2)
        min_dist2[idx] = 0.0

    pts_filtered = pts[selected_idx]

    # neighborhood search
    k = 20
    tree = cKDTree(pts)
    query_pts = tree.query(pts_filtered,k=k+1)
    query_pts = query_pts[1]
    query_self = query_pts[:,0]
    query_pts = query_pts[:,1:]

    # normals estimation

    features = data.x[query_pts]
    #bc_norm = minmax_norm(features[:,:,:3])
    #fext_norm = minmax_norm(features[:,:,3:])
    bc_std = torch.std(features[:,:,:3],dim=1)
    fext_std = torch.std(features[:,:,3:],dim=1)
    std_sum = torch.mean(fext_std,dim=1) #standard deviations of BCs + forces (0-1 normalized)
    bc_std_sum = torch.mean(bc_std,dim=1)
    
    rel_pos = data.pos[query_pts]-(data.pos[query_self]).unsqueeze(1).repeat(1, k, 1)
    rel_pos = minmax_norm(rel_pos)
    relpos_std = torch.std(rel_pos,dim=1)
    pos_std_sum = torch.mean(relpos_std,dim=1) #standard deviations of relative positions
    #feature_norm = torch.stack([std_sum,pos_std_sum],dim=1) # geometry & fixity & external forces -> based on neighborhood
    #feature_norm = torch.sum(feature_norm,axis=1)

    # mask by feature and position variance
    sig_mask_bc = (bc_std_sum > torch.quantile(bc_std_sum, 0.95))
    mid_mask_bc = (bc_std_sum < torch.quantile(bc_std_sum, 0.05)) # if any, usually will return no points
    sig_idx_bc = torch.where(sig_mask_bc | mid_mask_bc)[0]

    sig_mask_feat = (std_sum > torch.quantile(std_sum, 0.95))
    mid_mask_feat = (std_sum < torch.quantile(std_sum, 0.15))
    sig_idx_in_feat = torch.where(sig_mask_feat | mid_mask_feat)[0]

    sig_mask_pos = (pos_std_sum > torch.quantile(pos_std_sum, 0.95))
    mid_mask_pos = (pos_std_sum < torch.quantile(pos_std_sum, 0.15))
    sig_idx_in_pos = torch.where(sig_mask_pos | mid_mask_pos)[0]          # positions in pts_filtered

    sig_idx_in_filtered = torch.unique(torch.cat([sig_idx_bc,sig_idx_in_feat,sig_idx_in_pos],dim=0))

    # map back to original node ids
    sig_idx_original = torch.as_tensor(selected_idx)[sig_idx_in_filtered]  # indices into data['nodes'].pos

    # use original-order points for the coarse set
    pts_significant = pts[sig_idx_original]

    # fixed BC => fixed also after aggregation
    fixity_mask = ((data.x[sig_idx_original, :3]).sum(dim=1) == 0)
    pts_free = pts_significant[fixity_mask]
    pts_fixed = pts_significant[~fixity_mask]

    # connect points via Delaunay tetrahedralization (local coarse ordering)
    tri = Delaunay(pts_significant)
    tets = tri.simplices
    tet_edges = np.array([[0, 1], [0, 2], [0, 3],
                          [1, 2], [1, 3], [2, 3]])
    edges = tets[:, tet_edges].reshape(-1, 2)
    edges_undirected = np.sort(edges, axis=1)
    edges_undirected = np.unique(edges_undirected, axis=0)

    ## Alpha shape for connections (invalid edges deleted)
    # get inside volume with original points
    # calculate % of given edge inside original shape
    # -> for each sampled point in line get nearest neighbors & distances of neighbors
    edges = torch.tensor(edges,dtype=torch.int)
    edge_start = pts_significant[edges[:,0]]
    edge_end = pts_significant[edges[:,1]]
    num_points = 20
    direction = edge_end - edge_start
    t = torch.linspace(0, 1, num_points, device=edge_start.device)  # distances along edge
    edge_sampled = edge_start.unsqueeze(-2) + t.unsqueeze(-1) * direction.unsqueeze(-2) # (N_edges, num_points, ndims)
    # get distances from sample points to nearest node
    edge_dists, _ = tree.query(edge_sampled[:,:],k=2)
    edge_dists = edge_dists[:,1:num_points-1,1] # exclude self
    #print(edge_dists.shape)
    #print(edge_dists)
    # calculate average distance
    query_dist,_ = tree.query(pts,k=2)
    query_dist = query_dist[:,1].mean(axis=0)
    #print('query dist:',query_dist)
    # eliminate outliers
    dist_sigma = 1
    dist_threshold = 0.05
    bad_sample = (edge_dists > query_dist*dist_sigma)
    bad_frac = bad_sample.mean(axis=1)
    keep_line = (bad_frac <= dist_threshold)
    edge_mask_lines   = keep_line
    #print(edge_mask_lines)

    # edge_attr in coarse (original) coordinates
    i_local = edges_undirected[:, 0]
    j_local = edges_undirected[:, 1]
    edge_attr = pts_significant[j_local] - pts_significant[i_local]  # same as before

    # map coarse-local edge endpoints back to original node ids
    edges_original_idx = torch.as_tensor(sig_idx_original, dtype=torch.long)[
        torch.as_tensor(edges_undirected, dtype=torch.long)
    ]  # shape [E, 2], indexes into data.pos

    # aggregate info
    tree_f = cKDTree(pts_free)
    dists, idx_nearest = tree_f.query(pts, k=1)
    groups = [data.x[idx_nearest == j] for j in range(len(pts_free))]

    pooled = torch.stack([g.sum(dim=0) for g in groups])
    pooled[:, :3] = 0 #aggregated forces

    # topological information
    tree_s = cKDTree(pts_significant)
    dist_topo, idx_topo = tree_s.query(pts,k=1)
    rel = [pts[idx_topo == j,:3]-point for j,point in enumerate(pts_significant)]

    mu = torch.stack([(r.mean(axis=0)) for r in rel])      # shape [num_coarse, 3]
    sigma = torch.stack([(r.std(axis=0)) for r in rel])
    topo = torch.cat([mu,sigma],dim=-1).to(torch.float32)

    coarse = torch.zeros((len(pts),pooled.size(-1)+topo.size(-1)))
    free_idx = sig_idx_original[fixity_mask]
    fixed_idx = sig_idx_original[~fixity_mask]
    coarse[free_idx, :6] = pooled
    coarse[fixed_idx, :3] = 1
    coarse[sig_idx_original, 6:] = topo

    #mu = torch.from_numpy()
    #sigma = torch.from_numpy(r.std(axis=1, ddof=0) for r in rel)

    return edges_original_idx.T, coarse

scaler = StandardScaler()
dataset_p = [build_laststep_features(d.clone()).to(device) for d in dataset]

for data_raw, data_proc in zip(dataset, dataset_p):
    #data_coarse = Data()
    extra_ei, agg_feat = coarsen(data_proc)  # extra_ei shape [2, E]
    src, dst = extra_ei
    extra_attr = (data_proc.pos[dst] - data_proc.pos[src]).float()

    ei_old = data_proc.edge_index
    ea_old = data_proc.edge_attr

    if ea_old.size(1) != extra_attr.size(1):
        extra_attr_full = torch.zeros((extra_attr.size(0), ea_old.size(1)),dtype=ea_old.dtype,device=ea_old.device)
        extra_attr_full[:, :extra_attr.size(1)] = extra_attr.to(ea_old.dtype)
    else:
        extra_attr_full = extra_attr.to(ea_old.dtype)

    # keep the existing graph, add new dimensions for coarse graph
    #data_raw.bc = agg_bc[:,:3] #fixed or not
    #data_raw.pos = #positions of nodes -> unchanged
    #data_raw.f_ext = agg_bc[:,3:] #external forces (aggregated)
    data_proc.coarse_features = agg_feat #0:3 => BC; 3:6 => f_ext; 6:12 => topology
    #data_raw.f_ts = #internal forces
    #data_raw.u_ts = #displacements time series
    data_proc.edge_index_coarse = extra_ei.to(ei_old.device)
    data_proc.edge_attr_coarse = extra_attr_full
    #data_raw.edge_index = torch.cat([ei_old, extra_ei.to(ei_old.device)], dim=1)
    #data_raw.edge_attr = torch.cat([ea_old, extra_attr_full], dim=0)

#torch.save(dataset, "../datasets/simple_beam/test_dataset.pt")

In [3]:
import torch
import torch.nn.functional as F
from torch.nn import ModuleList

from torch_geometric.nn import pool
import torch_geometric.transforms as T
from torch_geometric.nn import MLP, GENConv, to_hetero, GCNConv

#device = torch.device('cuda:1')
torch.set_default_dtype(torch.float32)
device = torch.device('cpu')

class GEN_Multiscale(torch.nn.Module):
    def __init__(self, in_channels,edge_in,layers,latent_dim, out_channels):
        super().__init__()
        self.proj = MLP(in_channels=in_channels,hidden_channels=latent_dim,out_channels=latent_dim,num_layers=2,act='leaky_relu',norm='layer')
        self.edge_proj = MLP(in_channels=edge_in,hidden_channels=latent_dim,out_channels=latent_dim,num_layers=2,act='leaky_relu',norm='layer')
        
        self.coarse_proj = MLP(in_channels=in_channels+6,hidden_channels=latent_dim,out_channels=latent_dim,num_layers=2,act='leaky_relu',norm='layer')
        self.coarse_edge_proj = MLP(in_channels=edge_in,hidden_channels=latent_dim,out_channels=latent_dim,num_layers=2,act='leaky_relu',norm='layer')
        
        self.layers = layers

        self.encoding_layers = ModuleList([
            GENConv(latent_dim * 2, latent_dim, norm='layer',msg_norm=True,edge_dim=latent_dim)
            for _ in range(layers)])
        
        self.coarse_layers = ModuleList([
            GENConv(latent_dim, latent_dim, norm='layer',msg_norm=True,edge_dim=latent_dim)
            for _ in range(layers)])

        self.inv_proj = MLP(in_channels=latent_dim,hidden_channels=latent_dim,out_channels=out_channels,num_layers=2,act='leaky_relu',plain_last=True)
    
    def forward(self,x,x_c,edge_index,edge_attr,edge_index_coarse,edge_attr_coarse):
        x = self.proj(x)
        x_c = self.coarse_proj(x_c)
        edge_attr = self.edge_proj(edge_attr)
        edge_attr_coarse = self.coarse_edge_proj(edge_attr_coarse)
        for conv in self.coarse_layers:
            m_c = conv(x_c, edge_index=edge_index_coarse, edge_attr=edge_attr_coarse)
            x_c = x_c+m_c
        for conv in self.encoding_layers:
            m = conv(torch.cat([x, x_c], dim=1), edge_index, edge_attr=edge_attr)
            x = x + m
        x = self.inv_proj(x)
        return x
    
class GCN_Multiscale(torch.nn.Module):
    def __init__(self, in_channels,edge_in,layers,latent_dim, out_channels):
        super().__init__()

        self.proj = MLP(in_channels=in_channels,hidden_channels=latent_dim,out_channels=latent_dim,num_layers=2,act='leaky_relu',norm='layer')
        self.edge_proj = MLP(in_channels=edge_in,hidden_channels=latent_dim,out_channels=latent_dim,num_layers=2,act='leaky_relu',norm='layer')
        self.layers = layers
        
        self.encoding_layers = ModuleList([
            GCNConv(latent_dim * 2, latent_dim, normalize=False)
            for _ in range(layers)])

        self.inv_proj = MLP(in_channels=latent_dim,hidden_channels=latent_dim,out_channels=out_channels,num_layers=2,act='leaky_relu',plain_last=True)
    
    def forward(self,x,edge_index,edge_attr, batch):
        x = self.proj(x)
        edge_weight = edge_attr.norm(dim=1)
        
        for conv in self.encoding_layers:
            x_global = pool.global_mean_pool(x,batch)
            x_expanded = x_global[batch]
            m = conv(torch.cat([x, x_expanded], dim=1), edge_index, edge_weight=edge_weight)
            x = F.silu(conv(torch.cat([x, x_expanded], dim=1), edge_index, edge_weight=edge_weight))
            x = x + m
        x = self.inv_proj(x)
        return x

In [4]:
def compute_losses(batch, pred):
    # Nodes
    y_u    = batch.y_u
    #y_fint = batch['nodes'].y_fint
    #fext   = batch['nodes'].x[:,3:6]
    #bc = batch['nodes'].x[:,:3]
    #pu  = pred['u'] 
    #print(pred)
    
    #pf = pred['fint']

    L_u    = F.mse_loss(pred, y_u,reduction='mean')
    #L_fint = F.mse_loss(pf, y_fint)
    #L_eq   = F.mse_loss(((torch.ones_like(bc)-bc)*(pf - fext)).sum(),torch.zeros((),device=bc.device))

    loss = L_u
    return loss, {'L_u': L_u.item()}#, 'L_fint': L_fint.item(), 'L_eq': L_eq.item(),


In [9]:
## Training Loop
#model = GNN(in_channels=6,edge_in=3,layers=12,latent_dim=128,out_channels=3).to(device)
model = GEN_Multiscale(in_channels=6,edge_in=3,layers=12,latent_dim=128,out_channels=3).to(device)

alias = "1224_multiscale"
opt = torch.optim.AdamW(model.parameters(), lr=5e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', factor=0.5, patience=300)

scaler.fit(dataset_p)
train_set = [scaler.transform(d) for d in dataset_p]

def run_epoch(loader, train=True):
    model.train(train)
    total, last_loss_dict = 0.0, {}
    for batch in loader:
        batch = batch.to(device)
        opt.zero_grad(set_to_none=True)
        #print(batch)
        #self,x,x_c,edge_index,edge_attr,edge_index_coarse,edge_attr_coarse

        coarse = batch.coarse_features
        coarse_mask = coarse.abs().sum(dim=1) > 0
        
        pred = model(
            batch.x[:,:6],
            batch.coarse_features,
            batch.edge_index,
            batch.edge_attr,
            batch.edge_index_coarse,
            batch.edge_attr_coarse)
        
        loss, loss_dict = compute_losses(batch, pred)
        last_loss_dict = loss_dict  # keep something to return

        if train:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

        total += loss.item()

    steps = max(1, len(loader))
    return total / steps, last_loss_dict


EPOCHS = 2000
best_val = float("inf")
best_state = None
loss_records = []

torch.cuda.empty_cache()
train_loader = DataLoader(train_set, batch_size=1, shuffle=True)

for epoch in range(1, EPOCHS + 1):
    train_loss, train_ld = run_epoch(train_loader, train=True)

    scheduler.step(train_loss)
    loss_records.append({"epoch": epoch, "train_loss": train_loss})

    if epoch == 1 or epoch % 5 == 0:
        print(f"Epoch {epoch:03d} | train {train_loss:.6f}")
    if epoch == 1 or epoch % 20 == 0:
        torch.save(model.state_dict(), f"../genconv/{alias}/weights_{epoch}.pt")

print("Best val:", best_val)

pd.DataFrame(loss_records).to_csv(f"../genconv/{alias}/losses.csv", index=False)

Epoch 001 | train 1.690812
Epoch 005 | train 0.964347
Epoch 010 | train 0.860543
Epoch 015 | train 0.792221
Epoch 020 | train 0.764925
Epoch 025 | train 0.780836
Epoch 030 | train 0.756914
Epoch 035 | train 0.765750
Epoch 040 | train 0.735661
Epoch 045 | train 0.751706
Epoch 050 | train 0.744847
Epoch 055 | train 0.759630
Epoch 060 | train 0.761347
Epoch 065 | train 0.738148
Epoch 070 | train 0.714574
Epoch 075 | train 0.805795
Epoch 080 | train 0.765133
Epoch 085 | train 0.732274
Epoch 090 | train 0.682177
Epoch 095 | train 0.750350
Epoch 100 | train 0.695922
Epoch 105 | train 0.657861
Epoch 110 | train 0.689245
Epoch 115 | train 0.624453
Epoch 120 | train 0.562317
Epoch 125 | train 0.665917
Epoch 130 | train 0.603517
Epoch 135 | train 0.552671
Epoch 140 | train 0.573122
Epoch 145 | train 0.527177
Epoch 150 | train 0.487129
Epoch 155 | train 0.679480
Epoch 160 | train 0.759388


KeyboardInterrupt: 